# Politics O2 — exploration notebook

Runs Phases A → B → C → D (with a focused `politics_d.py` for the headline
strategy) on the Politics tag, then plots cumulative PnL and exposure over
time, in the style of `stage1_scaled.ipynb`.

All artefacts are written to this folder. To re-run, execute the cells in
order. The pipeline is small-first (CLAUDE.md): the runner scripts accept
`--max-shards` and are sample-tested before a full run.

In [1]:
# Setup: imports, paths
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name != "polymarket" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
PROJECT = ROOT
NOTEBOOKS = PROJECT / "notebooks"
WALLET = NOTEBOOKS / "wallet_selection"
SIGNAL_LAB = WALLET / "signal_lab"
HERE = SIGNAL_LAB / "onchain" / "politics"

for p in (str(PROJECT), str(NOTEBOOKS), str(WALLET)):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"PROJECT: {PROJECT}")
print(f"HERE:    {HERE}")
print(f"PYTHON:  {sys.executable}")

PROJECT: /Users/vobornij/projects/polymarket
HERE:    /Users/vobornij/projects/polymarket/notebooks/wallet_selection/signal_lab/onchain/politics
PYTHON:  /Users/vobornij/projects/polymarket/.venv/bin/python


## Phase A — control: existing composite on COPY_DEFAULT

Same as the Weather run: 5 strategies (CopyCrowdEntryTiming, FadeReactiveSellFlow,
UwlOppContrarian, FreshOppositeCrowdingFilter, GamblerCapitulationSqueeze) on the
COPY_DEFAULT-selected wallet subset.

In [ ]:
# Run Phases A, B, C, D in parallel via threads. Data is loaded once
# (cached in the o2_runner module) and shared across all phases. Phase D
# reuses Phase A's composite_shrinkage_markowitz (no re-run of the 5
# strategies on COPY_DEFAULT). Switch MAX_SHARDS=2 for a sample run.
import time
from concurrent.futures import ThreadPoolExecutor
import numpy as np

from signal_lab.onchain import o2_runner
from signal_lab.onchain.o2_runner import phase_a, phase_b, phase_c
from signal_lab.onchain.politics import politics_d as _pd_mod
from signal_lab.onchain.politics.politics_d import (
    _attach_composite_and_rule, BUDGET, _save_daily_pnl,
)
from signal_lab.sizing import capital_constrained_sim, select_scale, sizing_sharpe
from signal_lab.signal_lib import compute_event_ic, spearman_rho

SPLIT_KWARGS = {
    "train_end": "2026-02-01",
    "val_end": "2026-05-31",
    "test_start": "2026-06-01",
}
MAX_SHARDS = None  # full data; set to 2 for a quick sample run
MAX_WORKERS = 4    # 4 phases; pandas releases the GIL on groupby/merge

# Point the runner's output dir at this folder so the per-phase
# CSVs/JSONs land in politics/, not in onchain/.
o2_runner.OUT_DIR = HERE
o2_runner._DATA_CACHE.clear()


def _run_phase_d(phase_a_norm, split_kwargs):
    """Phase D body — reuses the Phase A composite (no COPY_DEFAULT re-run)."""
    t0 = time.time()
    splits = _attach_composite_and_rule(
        {}, split_kwargs=split_kwargs, phase_a_norm=phase_a_norm,
    )
    rows = []
    for split in ("train", "val", "test"):
        frame = splits[split]
        rows.append({
            "split": split,
            "IC_target_combo": compute_event_ic(frame["composite"], frame["copyable_pnl"]),
            "IC_pnl_res_combo": compute_event_ic(frame["composite"], frame["pnl_res"]),
            "IC_roi_res_combo": compute_event_ic(frame["composite"], frame["roi_res"]),
            "IC_target_rule": compute_event_ic(frame["rule_mask"], frame["copyable_pnl"]),
            "IC_pnl_res_rule": compute_event_ic(frame["rule_mask"], frame["pnl_res"]),
            "IC_roi_res_rule": compute_event_ic(frame["rule_mask"], frame["roi_res"]),
            "spearman_price_combo": spearman_rho(frame["composite"], frame["price"]),
            "spearman_price_rule": spearman_rho(frame["rule_mask"], frame["price"]),
            "n": int(len(frame)),
        })
    df_ic = pd.DataFrame(rows)
    df_ic.to_csv(HERE / "o2_d_politics_composite.csv", index=False)

    scale_grid = np.arange(0.1, 3.01, 0.1)
    siz_rows = []
    for split in ("val", "test"):
        frame = splits[split]
        best_scale, _ = select_scale(frame, "composite", BUDGET, scale_grid, 0.0, primary="sharpe_daily")
        res = capital_constrained_sim(frame, "composite", BUDGET, float(best_scale), 0.0)
        daily = res["daily_pnl"]
        siz_rows.append({
            "split": split, "scale": float(best_scale),
            "trades": int(res["trades"]), "net_pnl": round(res["net_pnl"], 2),
            "peak_used": round(res["peak_used"], 2),
            "mean_used": round(res["mean_used"], 2),
            "pnl_per_peak": round(res["net_pnl"] / max(res["peak_used"], 1e-9), 4),
            "sharpe_daily": round(sizing_sharpe(daily, 365.0), 3),
            "n_candidates": int(len(frame)),
            "score_col": "composite",
        })
        best_rule_scale, _ = select_scale(frame, "rule_mask", BUDGET, scale_grid, 0.0, primary="sharpe_daily")
        res_rule = capital_constrained_sim(frame, "rule_mask", BUDGET, float(best_rule_scale), 0.0)
        siz_rows.append({
            "split": split, "scale": float(best_rule_scale),
            "trades": int(res_rule["trades"]), "net_pnl": round(res_rule["net_pnl"], 2),
            "peak_used": round(res_rule["peak_used"], 2),
            "mean_used": round(res_rule["mean_used"], 2),
            "pnl_per_peak": round(res_rule["net_pnl"] / max(res_rule["peak_used"], 1e-9), 4),
            "sharpe_daily": round(sizing_sharpe(res_rule["daily_pnl"], 365.0), 3),
            "n_candidates": int(len(frame)),
            "score_col": "rule_mask",
        })
    df_siz = pd.DataFrame(siz_rows)
    df_siz.to_csv(HERE / "o2_d_politics_sizing.csv", index=False)
    summary = {
        "tag": "Politics",
        "rule": "price_lt_0p1",
        "a_composite_col": "composite_shrinkage_markowitz",
        "split_ic": df_ic.to_dict(orient="records"),
        "sizing": df_siz.to_dict(orient="records"),
    }
    (HERE / "o2_d_politics_summary.json").write_text(
        json.dumps(summary, indent=2, default=str)
    )
    print(f"[D/Politics] IC table:")
    print(df_ic.round(4).to_string(index=False))
    print(f"[D/Politics] sizing:")
    print(df_siz.round(4).to_string(index=False))
    _save_daily_pnl(splits)
    print(f"[D/Politics] done in {time.time() - t0:.1f}s")
    return summary


t0 = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    fut_a = ex.submit(phase_a, "Politics", MAX_SHARDS, split_kwargs=SPLIT_KWARGS)
    # Phase B is intentionally a 2-shard sample (full data timed out
    # historically on Politics all-buyers). The cache key differs by
    # max_shards, so this is a separate (small) load.
    fut_b = ex.submit(phase_b, "Politics", 2, split_kwargs=SPLIT_KWARGS)
    fut_c = ex.submit(phase_c, "Politics", MAX_SHARDS, split_kwargs=SPLIT_KWARGS)

    # Phase D depends on Phase A's composite. Block on A first, then
    # submit D; it overlaps with whatever is still running from B/C.
    summary_a = fut_a.result()
    t_a = time.time() - t0
    print(f"\n>>> Phase A done in {t_a:.1f}s")
    fut_d = ex.submit(_run_phase_d, summary_a["_normalized"], SPLIT_KWARGS)

    summary_b = fut_b.result()
    summary_c = fut_c.result()
    summary_d = fut_d.result()

PHASE_A_SUMMARY = summary_a
PHASE_D_SUMMARY = summary_d
print(f"\n>>> All 4 phases done in {time.time() - t0:.1f}s (was ~12 min sequentially)")
print(f"    A: {summary_a['n_candidates']} candidates, "
      f"B: {summary_b['n_candidates']} candidates, "
      f"C: {len(summary_c['rules'])} rules, "
      f"D: {summary_d['n_components']} components")


In [3]:
a_csv = HERE / "o2_a_politics_composite.csv"
a_df = pd.read_csv(a_csv)
a_pivot = a_df.pivot_table(index="scheme", columns="split",
                          values=["IC_target", "IC_pnl_res", "spearman_price"])
print("Phase A composite ICs (control on COPY_DEFAULT):")
print(a_pivot.round(4).to_string())

a_summary = json.loads((HERE / "o2_a_politics_summary.json").read_text())
print(f"\nN COPY_DEFAULT wallets: {a_summary['n_wallets_selected']}")
print(f"N candidate signals:    {a_summary['n_candidates']}")

Phase A composite ICs (control on COPY_DEFAULT):
                    IC_pnl_res                 IC_target                 spearman_price                
split                     test   train     val      test   train     val           test   train     val
scheme                                                                                                 
equal                   0.0340  0.1492  0.0344    0.0339  0.2039  0.0281        -0.0482  0.1397 -0.0715
ic_weighted             0.0416  0.1564  0.0380    0.0644  0.2493  0.0655         0.0176  0.2395  0.0211
shrinkage_markowitz     0.0335  0.1524  0.0469    0.1630  0.3362  0.1815         0.3240  0.4902  0.3301

N COPY_DEFAULT wallets: 107
N candidate signals:    15


## Phase B — non-copy-trade baseline (ALL_BUYERS mask)

Same 5 strategies but with `copy_mask = ALL_BUYERS` (no quality filter). The
Phase B on full Politics data is **expensive** (1.9M opening BUYs × 15
position signals), so the runner will likely time out. We use the 2-shard
sample as a reference; the per-signal ICs are still informative for the
UWL_OPP family which dominates.

In [5]:
b_per = pd.read_csv(HERE / "o2_b_politics_per_signal.csv")
b_per = b_per.assign(_abs=b_per["IC_val"].abs()).sort_values(
    "_abs", ascending=False).drop(columns="_abs")
print("Phase B per-signal ICs (sample):")
print(b_per.head(10).round(4).to_string(index=False))

Phase B per-signal ICs (sample):
                    signal  IC_train  IC_val  IC_test
        sig_uwl_opp_retail    0.3131  0.2997   0.1830
         sig_uwl_opp_whale    0.2551  0.1943   0.2948
    sig_fsf_sell_6h_retail    0.0158 -0.1442  -0.0544
sig_fsf_sell_6h_both_sides    0.0541 -0.0956  -0.0125
    sig_val_opp_overseller   -0.2441 -0.0870  -0.0784
sig_fsf_sell_6h_overseller    0.1295 -0.0822  -0.0036
       sig_val_opp_gambler    0.0227  0.0796  -0.0075
       sig_uwl_opp_gambler    0.2053  0.0643   0.1932
    sig_val_opp_both_sides   -0.1613 -0.0616  -0.0706
       sig_val_opp_flipper   -0.1183 -0.0169  -0.0645


## Phase C — direct price / lead / market rules

Six simple boolean rules evaluated against `roi_res` on the all-buyers
frame. The headline rule on Politics is `price_lt_0p1` (buy every opening
BUY with `price < 0.1`).

In [7]:
c_summary = pd.read_csv(HERE / "o2_c_politics_summary.csv")
print("Phase C rule ICs (full data):")
print(c_summary.round(4).to_string(index=False))

Phase C rule ICs (full data):
                    rule  IC_train  IC_val  IC_test  val_fire_rate  test_fire_rate  val_fires  test_fires  same_sign_val_test
price_lt_0p5_lead_gt_24h   -0.0363 -0.0308   0.0490         0.4288          0.4350    1027125      507141               False
price_lt_0p5_lead_gt_72h   -0.0272 -0.0349   0.0261         0.3162          0.3315     757369      386475               False
            price_lt_0p1    0.0208  0.0966   0.1683         0.2425          0.2378     580885      277292                True
            price_gt_0p9   -0.0023  0.0418   0.0436         0.1956          0.1553     468570      181063                True
   price_mid_lead_gt_24h   -0.0045 -0.0545  -0.0932         0.1989          0.2401     476461      279923                True
        recurring_market   -0.0256 -0.0370  -0.0431         0.9499          0.9747    2275390     1136412                True


## Phase D — combine Phase A composite with the Phase C rule

Avoids the slow full-universe `run_strategies` call. Reuses the Phase A
`shrinkage_markowitz` composite, the `price_lt_0p1` rule, and runs capital-
constrained sizing on the all-buyers frame. Also reports the rule-only
sizing as a separate baseline (Strategy P-1).

In [9]:
d_ic = pd.read_csv(HERE / "o2_d_politics_composite.csv")
d_siz = pd.read_csv(HERE / "o2_d_politics_sizing.csv")
print("Phase D ICs (composite = A_score + rule_mask, sign-only):")
print(d_ic.round(4).to_string(index=False))
print("\nPhase D sizing ($10k budget, val-tuned scale):")
print(d_siz.round(4).to_string(index=False))

Phase D ICs (composite = A_score + rule_mask, sign-only):
split  IC_target_combo  IC_pnl_res_combo  IC_roi_res_combo  IC_target_rule  IC_pnl_res_rule  IC_roi_res_rule  spearman_price_combo  spearman_price_rule       n
train          -0.3048            0.0058            0.0108         -0.3493          -0.0055           0.0208               -0.6484              -0.7226 3055403
  val          -0.2397            0.0787            0.0862         -0.2683           0.0787           0.0961               -0.6752              -0.7408 2417922
 test          -0.2300            0.1019            0.1538         -0.2490           0.1014           0.1677               -0.6948              -0.7368 1170622

Phase D sizing ($10k budget, val-tuned scale):
split  scale  trades   net_pnl  peak_used  mean_used  pnl_per_peak  sharpe_daily  n_candidates score_col
  val    0.2  157919 181830.05    10000.0    3076.24       18.1830         0.459       2417922       NaN
  val    0.2  188564 161582.64    10000.0   

## Verify

Run the per-phase verifier for this folder:

In [ ]:
# Run all 4 phase verifiers in parallel (each is just a file check).
from concurrent.futures import ThreadPoolExecutor
from signal_lab.onchain import verify_o2 as _verify

def _v(ph):
    print(f"\n=== Politics/{ph} ===", flush=True)
    ok = _verify.PHASES[ph]("Politics", HERE)
    return ph, ok

with ThreadPoolExecutor(max_workers=4) as ex:
    results = list(ex.map(_v, ["a", "b", "c", "d"]))
for ph, ok in results:
    if not ok:
        raise SystemExit(f"verify {ph} failed")


## Plots — PnL (trade vs resolution attribution) & exposure (test split)

Mirrors `stage1_scaled.ipynb`: cumulative copyable PnL is shown twice —
attributed at **trade time** (`dt`) and at **contract resolution time**
(`last_condition_trade_ts`). Exposure opens at each BUY (`qty = scale *
max(0, score) * copyable_qty`, capped at `copyable_qty`, at `price`) and
closes only at resolution for contracts resolved within the test window,
so exposure still open on unresolved contracts does not drop to 0.


In [ ]:
# Reuse the data + Phase A composite already in memory (from the
# parallel-run cell above). No subprocess, no data reload.
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from signal_lab.sizing import capital_constrained_sim, select_scale
from signal_lab.onchain.politics.politics_d import BUDGET

splits = _attach_composite_and_rule(
    {}, split_kwargs=SPLIT_KWARGS,
    phase_a_norm=PHASE_A_SUMMARY["_normalized"],
)
test = splits["test"]
val = splits["val"]

scale_grid = np.arange(0.1, 3.01, 0.1)
per_config = {}
for col, label in (("composite", "combo"), ("rule_mask", "rule")):
    best_scale, _ = select_scale(
        val, col, BUDGET, scale_grid, 0.0, primary="sharpe_daily"
    )
    res = capital_constrained_sim(test, col, BUDGET, float(best_scale), 0.0)
    taken_idx = res["taken"].values if hasattr(res["taken"], "values") else res["taken"]
    rows = test.loc[pd.Index(taken_idx)].copy() if len(taken_idx) else test.iloc[0:0].copy()
    if rows.empty:
        print(f"  test {col}: no taken trades")
        continue

    # Same qty formula as the sim: qty = clip(scale * max(0, score) * copyable_qty, 0, copyable_qty).
    qty = np.clip(
        float(best_scale) * np.clip(rows[col].fillna(0.0), 0, None) * rows["copyable_qty"],
        0.0, rows["copyable_qty"],
    )
    rows["qty"] = qty
    rows["copy_pnl"] = rows["copyable_pnl"] / rows["copyable_qty"].replace(0, np.nan) * qty
    rows["notional"] = rows["qty"] * rows["price"]

    rows["res_ts"] = pd.to_datetime(rows["last_condition_trade_ts"], utc=True, errors="coerce")
    window_end = rows["dt"].max()
    rows["resolved"] = rows["res_ts"] <= window_end

    open_ev = pd.DataFrame({"ev_dt": rows["dt"], "exposure_delta": rows["notional"]})
    close_ev = pd.DataFrame({
        "ev_dt": rows.loc[rows["resolved"], "res_ts"],
        "exposure_delta": -rows.loc[rows["resolved"], "notional"],
    })
    ev = pd.concat([open_ev, close_ev], ignore_index=True).sort_values("ev_dt").reset_index(drop=True)
    ev["exposure"] = ev["exposure_delta"].cumsum()

    pnl_trade = (
        rows[["dt", "copy_pnl"]].rename(columns={"dt": "ev_dt"})
        .sort_values("ev_dt").reset_index(drop=True)
    )
    pnl_trade["cum_pnl"] = pnl_trade["copy_pnl"].cumsum()

    pnl_res = (
        rows.loc[rows["resolved"], ["res_ts", "copy_pnl"]]
        .rename(columns={"res_ts": "ev_dt"})
        .sort_values("ev_dt").reset_index(drop=True)
    )
    pnl_res["cum_pnl"] = pnl_res["copy_pnl"].cumsum()

    per_config[label] = {
        "scale": float(best_scale),
        "exposure": ev,
        "pnl_trade": pnl_trade,
        "pnl_res": pnl_res,
        "n_taken": int(len(rows)),
        "n_resolved": int(rows["resolved"].sum()),
    }
    print(
        f"  test {col}: scale={best_scale:.2f} taken={len(rows):,} "
        f"resolved={int(rows['resolved'].sum()):,} "
        f"peak_exposure={ev['exposure'].max():,.0f} "
        f"ending_exposure={ev['exposure'].iloc[-1]:,.0f}"
    )

print(f"\nconfigs: {list(per_config)}")


In [ ]:
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
    row_heights=[0.4, 0.3, 0.3],
    subplot_titles=(
        "Cumulative copyable PnL — attributed at trade time",
        "Cumulative copyable PnL — attributed at contract resolution",
        "Exposure (USDC; closes at resolution for resolved contracts)",
    ),
)
COLOURS = {"combo": "#1f77b4", "rule": "#ff7f0e"}
for label, cfg in per_config.items():
    for row, key, trace_name in (
        (1, "pnl_trade", "cum pnl (trade time)"),
        (2, "pnl_res", "cum pnl (resolution time)"),
        (3, "exposure", "exposure"),
    ):
        fr = cfg[key]
        y = fr["cum_pnl"] if key != "exposure" else fr["exposure"]
        fig.add_trace(
            go.Scattergl(
                x=fr["ev_dt"], y=y, mode="lines",
                name=f"{label} · {trace_name}", legendgroup=label,
                line=dict(color=COLOURS[label]),
                hovertemplate="%{x|%Y-%m-%d}<br>%{y:,.0f}<extra>"
                              + f"{label} · {trace_name}" + "</extra>",
            ),
            row=row, col=1,
        )

fig.update_layout(
    template="plotly_dark",
    title="Politics O2 — PnL (trade vs resolution) & exposure (test split)",
    height=900,
    legend_title="config",
    hovermode="x unified",
)
fig.update_yaxes(title_text="cumulative pnl (USDC)", row=1, col=1)
fig.update_yaxes(title_text="cumulative pnl (USDC)", row=2, col=1)
fig.update_yaxes(title_text="exposure (USDC)", row=3, col=1)
fig.update_xaxes(title_text="date", row=3, col=1)
fig.show()


## Headline summary

Train ends 2026-02-01, val ends 2026-05-31, **test starts 2026-06-01**
(the split the user asked for). Two simple working strategies on
Politics under this split:

1. **Strategy P-1: `price_lt_0p1`** — buy every opening BUY with `price < 0.1`.
   Test IC (vs `roi_res`): +0.168. Sizing on test: net PnL +$9,601 on a $10k
   budget, 65,453 fired trades, daily Sharpe 0.32.
2. **Strategy P-2: composite (Phase A `shrinkage_markowitz` + rule mask)** —
   sign-only sum of the 5-strategy composite and the `price_lt_0p1` mask.
   Test IC (vs `roi_res`): +0.154. Sizing on test: +$1,667 on $10k, 60,638
   fired trades, daily Sharpe 0.09.

Both pass the O2 loose gate (`|val IC| > 0.005` same sign on test). They are
uncorrelated with the Weather composite by construction (different tags,
no shared trades), so they would be additive in a multi-tag portfolio.

The plots show cumulative copyable PnL twice — attributed at **trade time**
(`dt`) and at **contract resolution time** (`last_condition_trade_ts`) —
plus aggregate **exposure** on the test split. Exposure opens at each BUY
(sized at the val-tuned scale) and closes at resolution only for contracts
resolved within the test window, so exposure still open on unresolved
contracts does not drop to 0.

See `o2_REPORT.md` at the parent `signal_lab/onchain/` for the cross-tag
comparison (Finance was tested in the same loop but the realised PnL is
small — external data is the missing piece for Finance).